# 02a · R0: ResNet-50 with the *current* recipe, on the leak-free folds

The recipe is copied from `cnn-ipynb/model_latest.ipynb`, unchanged: ImageNet-V2 weights, plain `Linear` head,
128x128 input, AutoAugment (ImageNet policy), AdamW lr 3e-4 / wd 1e-4, cosine schedule stepped per epoch
(T_max = 15), 15 epochs, batch 32, checkpoint = first epoch with the best **validation accuracy** (as in the
original notebook; no tie-break). One epoch = one pass over the
published files of the training groups (byte copies included, as the original notebook iterated over all files).

**Only the split changes:** group-aware folds from notebook 01 instead of a random file split.

Time: about 25 min per fold on an RTX 3070 Ti Laptop (≈ 2 h for 5 folds). Safe to stop and re-run: finished
folds are skipped.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "PLAN.md").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from common import config, data, env
from common.cv import run_cv
from common.train import TrainConfig

config.ensure_dirs()
device = env.require_cuda()
env.seed_everything()
uniq, splits = data.load_unique(), data.load_splits()
images = data.load_images(uniq)
print(images.shape, images.dtype)

In [ ]:
R0 = TrainConfig(
    name="R0", arch="resnet50_linear", size=128, train_aug="autoaugment", sampler="copies", samples_per_epoch=None,
    epochs=15, batch_size=32, optimizer="adamw", lr=3e-4, weight_decay=1e-4, backbone_lr_mult=1.0,
    scheduler="cosine_epoch", label_smoothing=0.0, ema_decay=None, mixup=False, select="val_acc", tie_break_loss=False,
    notes="Recipe of cnn-ipynb/model_latest.ipynb, unchanged; only the split is group-aware.")
summary = run_cv(R0, device, uniq, splits, images)

In [ ]:
import pandas as pd

pd.DataFrame({k: {"best_epoch": v["best_epoch"], "minutes": round(v["train_minutes"], 1),
                  "test_acc": v["test"]["accuracy"], "test_macro_f1": v["test"]["macro_f1"]}
              for k, v in summary["folds"].items()}).T